In [1]:
%pip install langchain_community
%pip install langchain_experimental
#register openai and get its api key
%pip install langchain-openai
%pip install chromadb #vector
%pip install langchain
%pip install beautifulsoup4 #web crawler

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.6/50.6 kB 610.3 kB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 47.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 28.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 408.0/408.0 kB 19.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 296.9/296.9 kB 15.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 76.4/76.4 kB 3.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 78.0/78.0 kB 3.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.5/49.5 kB 1.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 144.5/144.5 kB 7.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 54.5/54.5 kB 2.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 58.3/58.3 kB 3.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 208.1/208.1 kB 4.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.9/49.

In [4]:
#don't show this
os.environ['OPENAI_API_KEY'] = 'your openai key'
os.environ['OPENAI_BASE_URL'] = 'your model url'
openai.api_key = os.environ['OPENAI_API_KEY']
openai.api_base = os.environ['OPENAI_BASE_URL']

In [3]:
import os
from langchain_community.document_loaders import WebBaseLoader
import bs4
import openai
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain import hub
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough #-> do nothing
import chromadb
from langchain_community.vectorstores import Chroma
from langchain_experimental.text_splitter import SemanticChunker #=> chop text into pieces by sematically
#how are you, I am find => "how are you", "I am find", => "how are", "you, I am", "find"

In [5]:
from langchain_core.runnables import Runnable
class TaskA(Runnable):
  def invoke(self, input, context=None):
    return input + " a"

class TaskBeautify(Runnable):
  def invoke(self, input, context=None):
    return input + " beautiful"

class TaskDay(Runnable):
  def invoke(self, input, context=None):
    return input + " day!"

task_a = TaskA()
task_beautiful = TaskBeautify()
task_day = TaskDay()

"|"
rag_chain = task_a | task_beautiful | task_day
res = rag_chain.invoke("what")
print(res)

class DoNothing(Runnable):
  def invoke(self, input, context=None):
    return input
# => RunnablePassThrough

what a beautiful day!


In [8]:
from langchain_core.runnables import RunnablePassthrough
input_data = {"msg": "this is a test"}
passthrough =  RunnablePassthrough()
res = passthrough.invoke(input_data)
print(res)

def context_func(input):
  return f"output from context func: {input}"

assgin = RunnablePassthrough().assign(context = context_func)
assign_res = assgin.invoke(input_data)
print(f"assign res: {assign_res}")

{'msg': 'this is a test'}
assign res: {'msg': 'this is a test', 'context': "output from context func: {'msg': 'this is a test'}"}


In [9]:
def task1(input):
  return f"task1: {input}"

def task2(input):
  return f"task2: {input}"

passthrough = RunnablePassthrough()
rag_chain = passthrough | task1 | task2
res = rag_chain.invoke("beginning....")
print(res)

task2: task1: beginning....


In [10]:
from langchain_core.runnables import RunnableParallel, RunnableMap

class Task1(Runnable):
  def invoke(self, input, context = None):
    return f"task1: {input}"

class Task2(Runnable):
  def invoke(self, input, context = None):
    return f"task2: {input}"

pipeline = RunnableParallel({
    "task1": Task1(),
    "task2": Task2(),
})

input_data = "input data"
output_data = pipeline.invoke("input_data")
print(output_data)

{'task1': 'task1: input_data', 'task2': 'task2: input_data'}


In [11]:
'''
scrap content from page by a url, after getting the content, we will extract some content with its element set with css class of
post-content, post-tile, post-header
'''
loader = WebBaseLoader(web_paths=("https://kbourne.github.io/chapter1.html",), bs_kwargs=dict(parse_only=bs4.SoupStrainer(
    class_=("post-content", "post-title", "post-header")
)))
docs = loader.load()
print(docs)

[Document(metadata={'source': 'https://kbourne.github.io/chapter1.html'}, page_content='\n\n      Introduction to Retrieval Augmented Generation (RAG)\n    \nDate: March 10, 2024  |  Estimated Reading Time: 15 min  |  Author: Keith Bourne\n\n  In the rapidly evolving field of artificial intelligence, Retrieval-Augmented Generation (RAG) is emerging as a significant addition to the Generative AI toolkit. RAG harnesses the strengths of Large Language Models (LLMs) and integrates them with internal data, offering a method to enhance organizational operations significantly. This book delves into the essential aspects of RAG, examining its role in augmenting the capabilities of LLMs and leveraging internal corporate data for strategic advantage.\nAs it progresses, the book outlines the potential of RAG in business, suggesting how it can make AI applications smarter, more responsive, and aligned with organizational objectives. RAG is positioned as a key facilitator of customized, efficient, 

In [12]:
#split huge text content into pieces
text_splitter = SemanticChunker(OpenAIEmbeddings())
splits = text_splitter.split_documents(docs)
print(splits)


[Document(metadata={'source': 'https://kbourne.github.io/chapter1.html'}, page_content="\n\n      Introduction to Retrieval Augmented Generation (RAG)\n    \nDate: March 10, 2024  |  Estimated Reading Time: 15 min  |  Author: Keith Bourne\n\n  In the rapidly evolving field of artificial intelligence, Retrieval-Augmented Generation (RAG) is emerging as a significant addition to the Generative AI toolkit. RAG harnesses the strengths of Large Language Models (LLMs) and integrates them with internal data, offering a method to enhance organizational operations significantly. This book delves into the essential aspects of RAG, examining its role in augmenting the capabilities of LLMs and leveraging internal corporate data for strategic advantage. As it progresses, the book outlines the potential of RAG in business, suggesting how it can make AI applications smarter, more responsive, and aligned with organizational objectives. RAG is positioned as a key facilitator of customized, efficient, a

In [13]:
#turn those pieces into vectors and save into the chroma db
vectorstore = Chroma.from_documents(documents = splits, embedding=OpenAIEmbeddings())
retriver = vectorstore.as_retriever()

In [14]:
query = "how dose RAG compare with fine-tuning"
relevant_docs = retriver.get_relevant_documents(query)
for doc in relevant_docs:
  print(doc.page_content)

<ipython-input-14-91d20bb5cfbd>:2: LangChainDeprecationWarning: The method `BaseRetriever.get_relevant_documents` was deprecated in langchain-core 0.1.46 and will be removed in 1.0. Use :meth:`~invoke` instead.
  relevant_docs = retriver.get_relevant_documents(query)


Can you imagine what you could do with all of the benefits mentioned above, but combined with all of the data within your company, about everything your company has ever done, about your customers and all of their interactions, or about all of your products and services combined with a knowledge of what a specific customer’s needs are? You do not have to imagine it, that is what RAG does! Even smaller companies are not able to access much of their internal data resources very effectively. Larger companies are swimming in petabytes of data that is not readily accessible or is not being fully utilized. Prior to RAG, most of the services you saw that connected customers or employees with the data resources of the company were really just scratching the surface of what is possible compared to if they could access ALL of the data in the company. With the advent of RAG and generative AI in general, corporations are on the precipice of something really, really big. Comparing RAG with Model Fi

In [15]:
#template of prompt
prompt = hub.pull("jclemens24/rag-prompt")
print(prompt)

input_variables=['context', 'question'] input_types={} partial_variables={} metadata={'lc_hub_owner': 'jclemens24', 'lc_hub_repo': 'rag-prompt', 'lc_hub_commit_hash': '1a1f3ccb9a5a92363310e3b130843dfb2540239366ebe712ddd94982acc06734'} messages=[HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['context', 'question'], input_types={}, partial_variables={}, template="You are an assistant for question-answering tasks. Use the following pieces of retrieved context to answer the question. If you don't know the answer, just say that you don't know.\nQuestion: {question} \nContext: {context} \nAnswer:"), additional_kwargs={})]


/usr/local/lib/python3.10/dist-packages/langsmith/client.py:354: LangSmithMissingAPIKeyWarning: API key must be provided when using hosted LangSmith API
  warnings.warn(


In [16]:
#helper function
def format_docs(docs):
  return "\n\n".join(doc.page_content for doc in docs)

llm = ChatOpenAI(model_name = "gpt-4o-mini", temperature=0)

In [ ]:

rag_chain = (
    {"context": retriver | format_docs,
     "question": RunnablePassthrough(), #basicall do nothing
    }
    | prompt | llm | StrOutputParser()
)

'''
rag_chain.invoke => retriver.invoke, RunnablePassthrough().invoke()

'''
response = rag_chain.invoke("How dose RAG compare with fine-tunning")
print(response)

Retrieval-Augmented Generation (RAG) and fine-tuning are two different approaches to enhancing the capabilities of Large Language Models (LLMs). 

RAG combines the strengths of LLMs with internal data, allowing organizations to access and utilize their own data effectively. It enhances the accuracy and relevance of responses by fetching specific information from databases in real time, thus expanding the model's knowledge beyond its initial training data. RAG is particularly useful for organizations that need to leverage their proprietary data or the latest information that was not included in the model's training set.

On the other hand, fine-tuning involves adjusting the weights and biases of a pre-trained model based on new training data. This process permanently alters the model's behavior and is suitable for teaching the model specialized tasks or adapting it to specific domains. However, fine-tuning can be complex and costly, especially as data sources evolve, and it may lead to 

In [18]:
def task_from_docs(input):
  print(f"task from docs for input: {input}")
  print(f"input['context']: {input['context']}")
  return lambda x : format_docs(input['context'])


rag_chain_from_docs = RunnablePassthrough().assign(context = task_from_docs)

rag_chain_with_source = RunnableParallel({
    "context": retriver,
    "question": RunnablePassthrough()
}).assign(answer = rag_chain_from_docs)

res = rag_chain_with_source.invoke("How dose RAG compare with fine-tuning")
print(f"context of res: {res['context']}")
print(f"question of res: {res['question']}")

rag_chain = rag_chain_with_source | prompt | llm | StrOutputParser()
response = rag_chain.invoke("How dose RAG compare with fine-tuning")

print(response)

task from docs for input: {'context': [Document(metadata={'source': 'https://kbourne.github.io/chapter1.html'}, page_content='Can you imagine what you could do with all of the benefits mentioned above, but combined with all of the data within your company, about everything your company has ever done, about your customers and all of their interactions, or about all of your products and services combined with a knowledge of what a specific customer’s needs are? You do not have to imagine it, that is what RAG does! Even smaller companies are not able to access much of their internal data resources very effectively. Larger companies are swimming in petabytes of data that is not readily accessible or is not being fully utilized. Prior to RAG, most of the services you saw that connected customers or employees with the data resources of the company were really just scratching the surface of what is possible compared to if they could access ALL of the data in the company. With the advent of RA